# Task 5 — Timing measurements
Czas enrollowania, weryfikacji 1:1, identyfikacji 1:N.

In [ ]:
import sys; sys.path.insert(0, '..')
import cv2, numpy as np
from pathlib import Path
from src.model import get_insightface_model
from src.database import EmbeddingDB
from src.enrollment import measure_enrollment_time
from src.authorization import measure_verify_time, measure_identify_time
from src.utils import list_images

app = get_insightface_model('buffalo_l', ctx_id=0)
db = EmbeddingDB.from_file()

# Wczytaj przykładowy obraz
test_imgs = list_images('../data/test')
sample_img = cv2.imread(str(test_imgs[0]))
sample_user = db.get_all_users()[0]

In [ ]:
N = 20
t_enroll = measure_enrollment_time(sample_img, app, db)
t_verify = measure_verify_time(sample_img, sample_user, app, db, n=N)
t_identify = measure_identify_time(sample_img, app, db, n=N)

print(f'Enrollment time:      {t_enroll*1000:.1f} ms')
print(f'Verify (1:1) time:    {t_verify*1000:.1f} ms  (avg over {N} runs)')
print(f'Identify (1:N) time:  {t_identify*1000:.1f} ms (avg over {N} runs, N={len(db)} users)')

In [ ]:
# Szacunek 1:N w zależności od rozmiaru bazy
import matplotlib.pyplot as plt
sizes = [10, 50, 100, 500, 1000]
# Zakładamy liniową zależność od liczby użytkowników
per_user_time = t_identify / len(db)
estimated = [s * per_user_time * 1000 for s in sizes]
fig, ax = plt.subplots()
ax.plot(sizes, estimated, marker='o')
ax.set_xlabel('Liczba enrolled użytkowników')
ax.set_ylabel('Szacowany czas identyfikacji [ms]')
ax.set_title('1:N Identification Time Estimate')
plt.tight_layout()
plt.savefig('../results/task5/timing_estimate.png', dpi=150)
plt.show()